In [2]:
import pandas as pd
import numpy as np
import timeit
import warnings
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.stats import pearsonr, spearmanr

# Вимикаємо зайві попередження для чистоти звіту
warnings.filterwarnings('ignore')

### Частина 2: Individual Household Electric Power Consumption
**1. Зчитування та Data Cleaning.**
Завантаження датасету, заміна пропущених значень (які в цьому датасеті позначені як '?') на `NaN` та їх видалення, конвертація типів даних.

In [5]:
def load_and_clean_data(filepath="household_power_consumption.txt"):
    print("Завантаження даних...")
    df = pd.read_csv(filepath, sep=';', low_memory=False, na_values=['?'])
    
    df = df.dropna()
    
    df['Time'] = pd.to_datetime(df['Time'], format='%H:%M:%S').dt.time
    
    df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y')
    
    return df

df_power = load_and_clean_data()
print(f"Дані успішно завантажено! Кількість записів після очищення: {len(df_power)}")
display(df_power.head())

Завантаження даних...
Дані успішно завантажено! Кількість записів після очищення: 2049280


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


**2.1 Обрати всі записи, у яких загальна активна споживана потужність перевищує 5 кВт.**
Профілювання часу виконання здійснюється за допомогою `timeit.default_timer()`.

In [6]:
def filter_power_gt_5(df):
    return df[df['Global_active_power'] > 5.0]

start_time = timeit.default_timer()
res_1 = filter_power_gt_5(df_power)
exec_time = timeit.default_timer() - start_time

print(f"Час виконання: {exec_time:.5f} секунд")
print(f"Знайдено записів: {len(res_1)}")
display(res_1.head())

Час виконання: 0.00445 секунд
Знайдено записів: 17547


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,2006-12-16,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,2006-12-16,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,2006-12-16,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


**2.2 Обрати всі записи, у яких сила струму лежить в межах 19-20 А, для них виявити ті, у яких пральна машина та холодильник (Sub_metering_2) споживають більше, ніж бойлер та кондиціонер (Sub_metering_3).**

In [7]:
def filter_intensity_and_submetering(df):
    filtered = df[(df['Global_intensity'] >= 19.0) & (df['Global_intensity'] <= 20.0)]
    result = filtered[filtered['Sub_metering_2'] > filtered['Sub_metering_3']]
    return result

start_time = timeit.default_timer()
res_2 = filter_intensity_and_submetering(df_power)
exec_time = timeit.default_timer() - start_time

print(f"Час виконання: {exec_time:.5f} секунд")
print(f"Знайдено записів: {len(res_2)}")
display(res_2.head())

Час виконання: 0.00668 секунд
Знайдено записів: 2509


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,2006-12-16,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,2006-12-17,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,2006-12-17,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,2006-12-17,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,2006-12-17,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


**2.3 Обрати випадковим чином 500000 записів (без повторів), для них обчислити середні величини усіх 3-х груп споживання електричної енергії.**

In [9]:
def random_sample_averages(df):
    sample_df = df.sample(n=500000, replace=False, random_state=42)
    
    averages = sample_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    return averages

start_time = timeit.default_timer()
res_3 = random_sample_averages(df_power)
exec_time = timeit.default_timer() - start_time

print(f"Час виконання: {exec_time:.5f} секунд")
print("Середні величини груп споживання:")
display(pd.DataFrame(res_3, columns=['Середнє значення (Вт год)']))

Час виконання: 0.16835 секунд
Середні величини груп споживання:


,Середнє значення (Вт год)
Sub_metering_1,1.119258
Sub_metering_2,1.308912
Sub_metering_3,6.452950


**2.4 Вибірка за часом та домінантним споживанням.**
- Час після 18:00
- Global_active_power > 6 кВт
- Група 2 (Sub_metering_2) є найбільшою серед трьох
- З першої половини відібраних беремо кожен 3-й результат, з другої - кожен 4-й.

In [10]:
import datetime

def complex_evening_filter(df):
    time_limit = datetime.time(18, 0, 0)
    
    step1 = df[(df['Time'] >= time_limit) & (df['Global_active_power'] > 6.0)]
    
    step2 = step1[(step1['Sub_metering_2'] > step1['Sub_metering_1']) & 
                  (step1['Sub_metering_2'] > step1['Sub_metering_3'])]
    
    half_idx = len(step2) // 2
    first_half = step2.iloc[:half_idx]
    second_half = step2.iloc[half_idx:]
    
    res_first = first_half.iloc[2::3]
    res_second = second_half.iloc[3::4]
    
    return pd.concat([res_first, res_second])

start_time = timeit.default_timer()
res_4 = complex_evening_filter(df_power)
exec_time = timeit.default_timer() - start_time

print(f"Час виконання: {exec_time:.5f} секунд")
print(f"Знайдено записів після усіх фільтрацій: {len(res_4)}")
display(res_4.head())

Час виконання: 0.08276 секунд
Знайдено записів після усіх фільтрацій: 308


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
43,2006-12-16,18:07:00,6.474,0.144,231.85,27.8,0.0,37.0,16.0
3007,2006-12-18,19:31:00,6.158,0.442,229.08,27.0,0.0,36.0,0.0
17497,2006-12-28,21:01:00,7.062,0.270,235.76,30.2,2.0,65.0,17.0
17500,2006-12-28,21:04:00,7.376,0.238,234.67,31.4,1.0,72.0,17.0
17503,2006-12-28,21:07:00,7.248,0.000,235.34,30.8,1.0,72.0,17.0


**3. Нормалізація, стандартизація, кореляція та One Hot Encoding.**
Для демонстрації беремо випадкові 10000 рядків (щоб не перевантажувати пам'ять під час візуалізації результатів).

In [12]:
def ml_preprocessing_steps(df):
    df_sample = df.sample(10000, random_state=42).copy()
    
    scaler_minmax = MinMaxScaler()
    scaler_std = StandardScaler()
    
    cols_to_scale = ['Global_active_power', 'Global_intensity']
    
    df_sample[['GAP_norm', 'GI_norm']] = scaler_minmax.fit_transform(df_sample[cols_to_scale])
    df_sample[['GAP_std', 'GI_std']] = scaler_std.fit_transform(df_sample[cols_to_scale])
    
    v1, v2 = df_sample['Global_active_power'], df_sample['Global_intensity']
    pearson_corr, _ = pearsonr(v1, v2)
    spearman_corr, _ = spearmanr(v1, v2)
    
    print(f"Кореляція Пірсона: {pearson_corr:.4f}")
    print(f"Кореляція Спірмена: {spearman_corr:.4f}\n")
    
    df_sample['Day_of_Week'] = df_sample['Date'].dt.day_name()
    
    df_encoded = pd.get_dummies(df_sample, columns=['Day_of_Week'], prefix='Day')
    
    return df_encoded

res_ml = ml_preprocessing_steps(df_power)
print("Вигляд датасету після нормалізації, стандартизації та OHE (One Hot Encoding):")
cols_to_show = ['Global_active_power', 'GAP_norm', 'GAP_std'] + [col for col in res_ml.columns if 'Day_' in col]
display(res_ml[cols_to_show].head())

Кореляція Пірсона: 0.9989
Кореляція Спірмена: 0.9955

Вигляд датасету після нормалізації, стандартизації та OHE (One Hot Encoding):


,Global_active_power,GAP_norm,GAP_std,Day_Friday,Day_Monday,Day_Saturday,Day_Sunday,Day_Thursday,Day_Tuesday,Day_Wednesday
1030580,1.502,0.168282,0.364874,False,True,False,False,False,False,False
1815,0.374,0.034980,-0.683891,False,False,False,True,False,False,False
1295977,0.620,0.064051,-0.455171,False,False,False,False,False,False,True
206669,0.280,0.023871,-0.771288,False,False,False,False,False,False,True
1048893,1.372,0.152919,0.244005,False,False,False,True,False,False,False
